In [1]:
# Importing Modules
import numpy as np
import pandas as pd
from src.dataloader import loadData
from src.model import GNNModel
from src.train_test import TrainGNN, TestGNN
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
######################## DATA-1 ##################################
# Loading ESOL data
esol_train_data = pd.read_csv("data/processed/train/ESOL.csv")
esol_val_data = pd.read_csv("data/processed/val/ESOL.csv")
esol_test_data = pd.read_csv("data/processed/test/ESOL.csv")

######################## DATA-2 ##################################
# Loading RT data
rt_train_data = pd.read_csv("data/processed/train/RT.csv")
rt_val_data = pd.read_csv("data/processed/val/RT.csv")
rt_test_data = pd.read_csv("data/processed/test/RT.csv")

######################## DATA-3 ##################################
# Loading Lipophilicity data
lipophilicity_train_data = pd.read_csv("data/processed/train/Lipophilicity.csv")
lipophilicity_val_data = pd.read_csv("data/processed/val/Lipophilicity.csv")
lipophilicity_test_data = pd.read_csv("data/processed/test/Lipophilicity.csv")

######################## DATA-4 ##################################
# Loading B3DB data
b3db_train_data = pd.read_csv("data/processed/train/B3DB.csv")
b3db_val_data = pd.read_csv("data/processed/val/B3DB.csv")
b3db_test_data = pd.read_csv("data/processed/test/B3DB.csv")

In [3]:
# Function to run analysis
def RunGNN(train_data, val_data, test_data, dataName, modelName, params):
    
	# Training data and target labels
	smiles_X_train = train_data["smiles"]
	X_train = smiles_X_train.to_numpy()
	y_train = train_data["target"]
	y_train = y_train.to_numpy()

    # Validation data and target labels
	smiles_X_val = val_data["smiles"]
	X_val = smiles_X_val.to_numpy()
	y_val = val_data["target"]
	y_val = y_val.to_numpy()
    
    # Testing data and target labels
	smiles_X_test = test_data["smiles"]
	X_test = smiles_X_test.to_numpy()
	y_test = test_data["target"]
	y_test = y_test.to_numpy()

    # Params
	h_dim, b_size, lr, d_out, wd = params

	# Loading dataset
	train_loader = loadData(X_train, y_train, batch_size=b_size)
	val_loader = loadData(X_val, y_val, batch_size=b_size)
	test_loader = loadData(X_test, y_test, batch_size=b_size)

	# Model
	model = GNNModel(num_features=6, hidden_dim=h_dim, model_type=modelName, dropout=d_out)

	# Model training
	model = TrainGNN(model, train_loader, val_loader, epochs=100, learning_rate=lr, w_decay=wd)

	# Model testing
	y_test, y_pred = TestGNN(model, test_loader)

	# Calculate MAE
	mae = mean_absolute_error(y_test, y_pred)

	# Calculate RMSE
	rmse = root_mean_squared_error(y_test, y_pred)
    
    # Bootstrap 95% CI
	rng = np.random.default_rng(42)
	mae_samples = []
	rmse_samples = []

	y_test_arr = np.array(y_test)
	y_pred_arr = np.array(y_pred)

	for _ in range(2000):
		idx = rng.integers(0, len(y_test_arr), len(y_test_arr))
		mae_samples.append(mean_absolute_error(y_test_arr[idx], y_pred_arr[idx]))
		rmse_samples.append(root_mean_squared_error(y_test_arr[idx], y_pred_arr[idx]))

	mae_lower, mae_upper = np.percentile(mae_samples, [2.5, 97.5])
	rmse_lower, rmse_upper = np.percentile(rmse_samples, [2.5, 97.5])

	# Return results
	return pd.DataFrame({ "Data": [dataName], "Model": [modelName],
                          "MAE": [mae], "MAE_lower": [mae_lower],
                          "MAE_upper": [mae_upper], "RMSE": [rmse],
                          "RMSE_lower": [rmse_lower], "RMSE_upper": [rmse_upper]
                        })

In [4]:
# Data dict
datasets = {"ESOL":{"train":esol_train_data, "val":esol_val_data, "test":esol_test_data},
            "Lipophilicity":{"train":lipophilicity_train_data, "val":lipophilicity_val_data, "test":lipophilicity_test_data},
            "RT":{"train":rt_train_data, "val":rt_val_data, "test":rt_test_data},
            "B3DB":{"train":b3db_train_data, "val":b3db_val_data,"test":b3db_test_data}
           }

# Models
models = ["GCN", "GAT", "GIN", "GraphSAGE"]

In [5]:
# List to store results
temp_out = []

# Loop for models
for modelName in models:
    # Loop for dataset
    for dataName, data in datasets.items():
        # Extract params
        temp_df = pd.read_csv(f"results/Output_Hyperparameter_Optimization_GNN_{dataName}.csv")
        temp = temp_df[temp_df["Model"]==modelName]
        params = temp.sort_values(by=["RMSE"]).head(1)[["h_dim", "b_size", "lr", "d_out", "w_decay"]].to_dict('records')[0].values()
        # Run Analysis for model and dataset
        temp_out.append(RunGNN(data["train"], data["val"], data["test"], dataName, modelName, params))

# Final output
GNN_out = pd.concat(temp_out, ignore_index=True)
GNN_out.to_csv("results/Output_GNN_Analysis.csv")
GNN_out

,Data,Model,MAE,MAE_lower,MAE_upper,RMSE,RMSE_lower,RMSE_upper
0,ESOL,GCN,0.803747,0.670405,0.947350,1.061003,0.864292,1.278594
1,Lipophilicity,GCN,0.878710,0.769154,0.995265,1.055461,0.926588,1.198384
2,RT,GCN,89.880458,75.849003,105.455456,118.113455,99.000354,138.305836
3,B3DB,GCN,0.430208,0.364747,0.506149,0.559038,0.480391,0.640310
4,ESOL,GAT,0.641924,0.533869,0.748612,0.842811,0.705996,0.981037
5,Lipophilicity,GAT,0.700531,0.606859,0.800789,0.865245,0.763755,0.966701
6,RT,GAT,66.634119,51.720306,82.690280,101.965631,74.722808,127.966217
7,B3DB,GAT,0.441033,0.368891,0.520848,0.588263,0.488169,0.684321
8,ESOL,GIN,0.716417,0.613794,0.822545,0.897771,0.766566,1.024533
9,Lipophilicity,GIN,0.779946,0.654446,0.923664,1.027574,0.868009,1.191816


In [6]:
# Function to plot barplot showing result
def plot_bar(data, target):
    data = data.copy()
    data["err_lower"] = data[target] - data[f"{target}_lower"]
    data["err_upper"] = data[f"{target}_upper"] - data[target]
    sns.set_theme(style="ticks", context="paper")
    model_order = data["Model"].unique()
    g = sns.catplot(
        data=data, kind="bar", x="Model", y=target, hue="Model",
        col="Data", col_wrap=4, sharey=False, height=4, width=0.8, 
        aspect=0.8, dodge=False, errorbar=None, order=model_order)
    for i, ax in enumerate(g.axes.flat):
        col_name = g.col_names[i]
        subdata = data[data["Data"] == col_name].set_index("Model").reindex(model_order).reset_index()
        if not subdata.dropna(subset=[target]).empty:
            ax.errorbar(
                x=np.arange(len(subdata)), 
                y=subdata[target],
                yerr=[subdata["err_lower"], subdata["err_upper"]], 
                fmt="none", capsize=4, linewidth=1, color="black")
    g.set_axis_labels("Model", f"{target}")
    g.set_titles("{col_name}")
    for ax in g.axes.flat:
        ax.tick_params(axis="x", rotation=40)
    plt.tight_layout()
    plt.savefig(f"results/GNN_Analysis_{target}_Plot.png", dpi=300)
    plt.close()

In [7]:
# Plotting barplot for RMSE
plot_bar(GNN_out, "RMSE")
# Plotting barplot for MAE
plot_bar(GNN_out, "MAE")